#  XEM TỬ VI BẰNG CHỈ TAY (PALMISTRY) - Mô hình CNN

## Mô tả dự án
Sử dụng **Convolutional Neural Network (CNN)** multi-output để phân tích ảnh chỉ tay và dự đoán **8 khía cạnh cuộc sống**:

| # | Khía cạnh | Mô tả |
|---|---|---|
| 1 |  **Tổng quan** | Vận mệnh chung |
| 2 |  **Tình duyên** | Tình cảm, hôn nhân |
| 3 |  **Công việc** | Sự nghiệp, thăng tiến |
| 4 |  **Sức khỏe** | Thể trạng, tuổi thọ |
| 5 |  **Tài chính** | Tài lộc, tiền bạc |
| 6 |  **Tương lai** | Triển vọng tương lai |
| 7 |  **Kiếp trước** | Tiền kiếp |
| 8 |  **Kiếp sau** | Lai sinh |

## 1. Import thư viện & Cấu hình môi trường

Cell này sẽ tự động phát hiện bạn đang chạy trên **Google Colab** hay **máy local**.
- Nếu trên Colab: sẽ liên kết với Google Drive của bạn. Bạn chỉ cần tải thư mục chứa ảnh lên Google Drive.
- Nếu trên local: sẽ dùng đường dẫn thư mục trực tiếp trên máy.

In [ ]:
import os, re, csv
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# === TỰ ĐỘNG PHÁT HIỆN MÔI TRƯỜNG ===
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('\n Đang chạy trên Google Colab!')
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Mặc định sử dụng thư mục "chi tay" trên MyDrive
    DATASET_PATH = '/content/drive/MyDrive/chi tay'
    os.makedirs(DATASET_PATH, exist_ok=True)
    
    print('\n HƯỚNG DẪN DÀNH CHO GOOGLE COLAB:')
    print('Hãy chắc chắn rằng bạn đã tải thư mục "chi tay" chứa các ảnh .jpg lên Google Drive của mình.')
    print(f'Thư mục hiện tại đang cấu hình là: {DATASET_PATH}')
else:
    print('\n Đang chạy trên máy local!')
    # === THAY ĐỔI ĐƯỜNG DẪN NÀY NẾU CẦN ===
    DATASET_PATH = r'c:\Users\MR ASUS\Downloads\chi tay'

CSV_PATH = os.path.join(DATASET_PATH, 'palmistry_labels.csv')
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 16
EPOCHS = 30
NUM_CLASSES = 3

# Tên các output
OUTPUT_NAMES = ['fortune_class', 'tinh_duyen', 'cong_viec', 'suc_khoe',
                'tai_chinh', 'tuong_lai', 'kiep_truoc', 'kiep_sau']

OUTPUT_LABELS_VN = {
    'fortune_class': ' Tổng Quan Vận Mệnh',
    'tinh_duyen':    ' Tình Duyên',
    'cong_viec':     ' Công Việc & Sự Nghiệp',
    'suc_khoe':      ' Sức Khỏe',
    'tai_chinh':     ' Tài Chính & Tài Lộc',
    'tuong_lai':     ' Tương Lai',
    'kiep_truoc':    ' Kiếp Trước',
    'kiep_sau':      ' Kiếp Sau',
}

# Kiểm tra thư mục
img_count = len([f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))])
print(f'\n📁 Trạng thái thư mục: {DATASET_PATH}')
print(f'🖼️ Số ảnh tìm thấy: {img_count}')
print(f'📄 CSV tồn tại: {os.path.exists(CSV_PATH)}')
if img_count == 0:
    print(" CẢNH BÁO: Không tìm thấy ảnh nào! Vui lòng kiểm tra lại đường dẫn hoặc upload dữ liệu.")

## 2. Tạo / Đọc dữ liệu nhãn CSV

Nếu file `palmistry_labels.csv` chưa tồn tại, cell này sẽ **tự động tạo** nhãn tử vi từ danh sách ảnh.

In [ ]:
# === TỰ ĐỘNG TẠO CSV NẾU CHƯA CÓ ===
if not os.path.exists(CSV_PATH) and img_count > 0:
    print(' File CSV chưa tồn tại → Đang tự động tạo nhãn...')
    
    image_files = sorted([f for f in os.listdir(DATASET_PATH)
                          if f.lower().endswith(('.jpg','.jpeg','.png'))])
    
    def get_pid(fname):
        m = re.match(r'(\d+)_IMG', fname)
        return int(m.group(1)) if m else 0
    
    def gen_labels(p):
        return {
            'life_line':  (p*7+3)%3,  'heart_line': (p*11+5)%3,
            'head_line':  (p*13+7)%3, 'fate_line':  (p*17+11)%3,
            'tinh_duyen': (p*19+2)%3, 'cong_viec':  (p*23+5)%3,
            'suc_khoe':   (p*29+7)%3, 'tai_chinh':  (p*31+11)%3,
            'tuong_lai':  (p*37+13)%3,'kiep_truoc': (p*41+17)%3,
            'kiep_sau':   (p*43+19)%3,
        }
    
    # Gán fortune_class cân bằng theo thứ tự person_id
    pids_unique = sorted(set(get_pid(f) for f in image_files))
    fortune_map = {pid: i % 3 for i, pid in enumerate(pids_unique)}
    
    columns = ['filename','person_id','life_line','heart_line','head_line','fate_line',
               'tinh_duyen','cong_viec','suc_khoe','tai_chinh','tuong_lai',
               'kiep_truoc','kiep_sau','fortune_class']
    
    rows = []
    for f in image_files:
        pid = get_pid(f)
        L = gen_labels(pid)
        fc = fortune_map[pid]
        rows.append([f, f'{pid:03d}', L['life_line'], L['heart_line'],
                     L['head_line'], L['fate_line'], L['tinh_duyen'],
                     L['cong_viec'], L['suc_khoe'], L['tai_chinh'],
                     L['tuong_lai'], L['kiep_truoc'], L['kiep_sau'], fc])
    
    with open(CSV_PATH, 'w', newline='', encoding='utf-8') as fout:
        writer = csv.writer(fout)
        writer.writerow(columns)
        writer.writerows(rows)
    
    print(f' Đã tạo {len(rows)} dòng dữ liệu → {CSV_PATH}')
elif img_count > 0:
    print(f' Đã tìm thấy file CSV: {CSV_PATH}')

# Đọc CSV nếu đã tồn tại
if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print(f"\nTổng số ảnh trong CSV: {len(df)} | Số người: {df['person_id'].nunique()}")
    print(f"Các cột dữ liệu: {list(df.columns)}")
    display(df.head(6))

In [ ]:
if 'df' in locals():
    # Thống kê phân bố tất cả nhãn
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    colors_map = {0: '#FF6B6B', 1: '#4ECDC4', 2: '#45B7D1'}
    
    for idx, col in enumerate(OUTPUT_NAMES):
        ax = axes[idx // 4][idx % 4]
        counts = df[col].value_counts().sort_index()
        bars = ax.bar(counts.index, counts.values,
                      color=[colors_map.get(i, '#gray') for i in counts.index],
                      edgecolor='black', linewidth=0.5)
        ax.set_title(OUTPUT_LABELS_VN[col], fontsize=11, fontweight='bold')
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(['Thấp', 'TB', 'Cao'], fontsize=9)
        for bar, val in zip(bars, counts.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                    str(val), ha='center', fontweight='bold', fontsize=10)
    
    plt.suptitle('Phân bố nhãn tất cả khía cạnh tử vi', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 3. Load và tiền xử lý ảnh

In [ ]:
images = []
valid_indices = []
skipped = []

if 'df' in locals():
    for idx, row in df.iterrows():
        img_path = os.path.join(DATASET_PATH, row['filename'])
        if not os.path.exists(img_path):
            skipped.append(row['filename'])
            continue
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, IMAGE_SIZE)
            images.append(img)
            valid_indices.append(idx)
        else:
            skipped.append(row['filename'])
    
    images = np.array(images, dtype=np.float32) / 255.0
    df_valid = df.iloc[valid_indices].reset_index(drop=True)
    
    print(f" Đã load thành công: {len(images)} ảnh")
    print(f" Bỏ qua: {len(skipped)} ảnh")
    if len(images) > 0:
        print(f"Kích thước dữ liệu: {images.shape}")

In [ ]:
if 'df_valid' in locals() and len(images) > 0:
    # Hiển thị ảnh mẫu theo nhóm fortune_class
    fortune_names_vn = ['Bình thường', 'Tốt', 'Rất tốt']
    fig, axes = plt.subplots(3, 5, figsize=(18, 12))
    
    for cls in range(3):
        cls_indices = np.where(df_valid['fortune_class'].values == cls)[0]
        samples = np.random.choice(cls_indices, min(5, len(cls_indices)), replace=False)
        for j, si in enumerate(samples):
            axes[cls][j].imshow(images[si])
            fname = df_valid.iloc[si]['filename'][:18]
            axes[cls][j].set_title(f'{fortune_names_vn[cls]}\n{fname}', fontsize=9)
            axes[cls][j].axis('off')
    
    plt.suptitle('Ảnh mẫu theo Tổng Quan Vận Mệnh', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 4. Chuẩn bị nhãn multi-output và chia dữ liệu

In [ ]:
if 'df_valid' in locals() and len(images) > 0:
    # Tạo dictionary nhãn cho multi-output
    all_labels = {}
    for col in OUTPUT_NAMES:
        all_labels[col] = df_valid[col].values
    
    # Chia dữ liệu: 80% Train, 10% Val, 10% Test
    indices = np.arange(len(images))
    idx_train, idx_temp = train_test_split(
        indices, test_size=0.2, random_state=42, stratify=all_labels['fortune_class']
    )
    idx_val, idx_test = train_test_split(
        idx_temp, test_size=0.5, random_state=42,
        stratify=all_labels['fortune_class'][idx_temp]
    )
    
    X_train, X_val, X_test = images[idx_train], images[idx_val], images[idx_test]
    
    y_train = {col: all_labels[col][idx_train] for col in OUTPUT_NAMES}
    y_val   = {col: all_labels[col][idx_val]   for col in OUTPUT_NAMES}
    y_test  = {col: all_labels[col][idx_test]  for col in OUTPUT_NAMES}
    
    print(f"Tập Train : {len(X_train)} ảnh")
    print(f"Tập Val   : {len(X_val)} ảnh")
    print(f"Tập Test  : {len(X_test)} ảnh")
    print(f"\nPhân bố fortune_class Train: {np.bincount(y_train['fortune_class'])}")
    print(f"Phân bố fortune_class Val  : {np.bincount(y_val['fortune_class'])}")
    print(f"Phân bố fortune_class Test : {np.bincount(y_test['fortune_class'])}")

## 5. Xây dựng mô hình CNN Multi-Output

In [ ]:
def build_multi_output_cnn(input_shape=(128, 128, 3), num_classes=3):
    """Xây dựng CNN multi-output cho dự đoán tử vi toàn diện."""
    
    inputs = layers.Input(shape=input_shape, name='input_image')
    
    # === Data Augmentation (chỉ hoạt động khi training) ===
    x = layers.RandomFlip('horizontal')(inputs)
    x = layers.RandomRotation(0.15)(x)
    x = layers.RandomZoom(0.15)(x)
    
    # === Block 1: 32 filters ===
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(32, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # === Block 2: 64 filters ===
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(64, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # === Block 3: 128 filters ===
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(128, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # === Block 4: 256 filters ===
    x = layers.Conv2D(256, (3, 3), padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(0.25)(x)
    
    # === Global Average Pooling ===
    x = layers.GlobalAveragePooling2D()(x)
    
    # === Shared Dense ===
    shared = layers.Dense(256)(x)
    shared = layers.BatchNormalization()(shared)
    shared = layers.Activation('relu')(shared)
    shared = layers.Dropout(0.5)(shared)
    
    shared = layers.Dense(128)(shared)
    shared = layers.BatchNormalization()(shared)
    shared = layers.Activation('relu')(shared)
    shared = layers.Dropout(0.5)(shared)
    
    # === 8 Output Heads ===
    outputs = {}
    for name in OUTPUT_NAMES:
        branch = layers.Dense(64, activation='relu', name=f'{name}_dense')(shared)
        branch = layers.Dropout(0.3)(branch)
        outputs[name] = layers.Dense(num_classes, activation='softmax', name=name)(branch)
    
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

model = build_multi_output_cnn(input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3), num_classes=NUM_CLASSES)

print(f"Tổng tham số: {model.count_params():,}")
print(f"Số output heads: {len(OUTPUT_NAMES)}")
print(f"\nCác output: {list(model.output_names)}")

In [ ]:
model.summary()

## 6. Biên dịch và Huấn luyện

In [ ]:
# Biên dịch: mỗi output dùng sparse_categorical_crossentropy
losses = {name: 'sparse_categorical_crossentropy' for name in OUTPUT_NAMES}

loss_weights = {
    'fortune_class': 2.0,
    'tinh_duyen':    1.0,
    'cong_viec':     1.0,
    'suc_khoe':      1.0,
    'tai_chinh':     1.0,
    'tuong_lai':     1.0,
    'kiep_truoc':    0.8,
    'kiep_sau':      0.8,
}

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=losses,
    loss_weights=loss_weights,
    metrics=['accuracy']
)

cb_early = callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
cb_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1)
cb_ckpt = callbacks.ModelCheckpoint(
    os.path.join(DATASET_PATH, 'best_palmistry_model.keras'),
    monitor='val_fortune_class_accuracy', save_best_only=True, verbose=1
)

print(" Mô hình đã biên dịch xong!")
print(f"   Epochs: {EPOCHS} | Batch: {BATCH_SIZE}")

In [ ]:
if 'X_train' in locals():
    print("🚀 Bắt đầu huấn luyện mô hình CNN Multi-Output...")
    print("=" * 60)
    
    history = model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=[cb_early, cb_lr, cb_ckpt],
        verbose=1
    )
    
    print("\n Huấn luyện hoàn tất!")
else:
    print(" Không có dữ liệu để huấn luyện. Vui lòng kiểm tra lại đường dẫn ảnh.")

## 7. Đánh giá kết quả huấn luyện

In [ ]:
if 'history' in locals():
    # Vẽ biểu đồ Accuracy cho tất cả output
    fig, axes = plt.subplots(2, 4, figsize=(22, 10))
    
    for idx, name in enumerate(OUTPUT_NAMES):
        ax = axes[idx // 4][idx % 4]
        acc_key = f'{name}_accuracy'
        val_acc_key = f'val_{name}_accuracy'
        
        if acc_key in history.history:
            ax.plot(history.history[acc_key], label='Train', linewidth=2, color='#2196F3')
            ax.plot(history.history[val_acc_key], label='Val', linewidth=2, color='#FF5722')
        ax.set_title(OUTPUT_LABELS_VN[name], fontsize=11, fontweight='bold')
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Accuracy')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(' Accuracy theo từng khía cạnh tử vi', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
if 'history' in locals():
    # Vẽ biểu đồ Loss tổng + bar chart accuracy
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2, color='#2196F3')
    axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2, color='#FF5722')
    axes[0].set_title(' Tổng Loss', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].legend(fontsize=12)
    axes[0].grid(True, alpha=0.3)
    
    # Bar chart val accuracy
    names_short = []
    val_accs = []
    for name in OUTPUT_NAMES:
        val_key = f'val_{name}_accuracy'
        if val_key in history.history:
            names_short.append(OUTPUT_LABELS_VN[name][:12])
            val_accs.append(history.history[val_key][-1])
    
    if val_accs:
        colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(val_accs)))
        axes[1].barh(names_short, val_accs, color=colors, edgecolor='black', linewidth=0.5)
        axes[1].set_xlim(0, 1)
        axes[1].set_xlabel('Validation Accuracy')
        axes[1].set_title(' Val Accuracy theo khía cạnh', fontsize=14, fontweight='bold')
        for i, v in enumerate(val_accs):
            axes[1].text(v + 0.01, i, f'{v:.2%}', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 8. Đánh giá trên tập Test

In [ ]:
if 'X_test' in locals():
    test_results = model.evaluate(X_test, y_test, verbose=0)
    y_pred_dict = model.predict(X_test, verbose=0)
    
    print("🎯 KẾT QUẢ TRÊN TẬP TEST")
    print("=" * 50)
    
    for name in OUTPUT_NAMES:
        y_true = y_test[name]
        y_pred = np.argmax(y_pred_dict[name], axis=1)
        acc = np.mean(y_true == y_pred)
        print(f"  {OUTPUT_LABELS_VN[name]:30s} → Accuracy: {acc:.2%}")

In [ ]:
if 'y_pred_dict' in locals():
    # Confusion Matrix cho fortune_class
    y_true_fc = y_test['fortune_class']
    y_pred_fc = np.argmax(y_pred_dict['fortune_class'], axis=1)
    
    cm = confusion_matrix(y_true_fc, y_pred_fc)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Bình thường', 'Tốt', 'Rất tốt'],
                yticklabels=['Bình thường', 'Tốt', 'Rất tốt'])
    plt.title('Ma trận nhầm lẫn - Tổng Quan Vận Mệnh', fontsize=14, fontweight='bold')
    plt.xlabel('Dự đoán')
    plt.ylabel('Thực tế')
    plt.tight_layout()
    plt.show()
    
    print("\nClassification Report - Tổng Quan Vận Mệnh:")
    print(classification_report(y_true_fc, y_pred_fc,
                                target_names=['Bình thường', 'Tốt', 'Rất tốt']))

## 9.  Hàm Dự đoán Tử Vi Chi Tiết

In [ ]:
# =====================================================
# TỪ ĐIỂN DỰ ĐOÁN TỬ VI CHI TIẾT BẰNG TIẾNG VIỆT
# =====================================================

PREDICTIONS_TEXT = {
    'fortune_class': {
        0: ["Vận mệnh đang ở giai đoạn tích lũy. Cuộc sống có những thử thách nhưng đây là cơ hội để rèn luyện ý chí. Đường chỉ tay cho thấy bạn là người kiên cường. Hãy kiên nhẫn, vận may sẽ đến.",
            "Tổng quan vận mệnh ở mức bình thường. Bạn cần nỗ lực nhiều hơn để đạt được mục tiêu. Nền tảng xây dựng bây giờ sẽ là bệ phóng cho tương lai rực rỡ."],
        1: ["Vận mệnh tốt! Đường chỉ tay cho thấy bạn có phước lành từ tiền kiếp. Cuộc sống thuận lợi, có quý nhân phù trợ. Sự nghiệp và tình cảm đều phát triển tích cực.",
            "Vận mệnh khá tốt, nhiều may mắn trong cuộc sống. Bạn có khả năng thu hút nguồn năng lượng tích cực. Các mối quan hệ hỗ trợ bạn rất nhiều."],
        2: ["Vận mệnh cực kỳ tốt!  Đường chỉ tay sâu, rõ ràng cho thấy bạn được trời phú cho nhiều phước lành. Mọi mặt cuộc sống đều hanh thông: sức khỏe dồi dào, tài lộc sung túc, tình duyên viên mãn.",
            "Vận mệnh rực rỡ! Bạn là người có mệnh phú quý, được nhiều người kính trọng. Cuộc đời nhiều thành tựu, gia đình hạnh phúc viên mãn."],
    },
    'tinh_duyen': {
        0: [" Tình duyên giai đoạn này còn nhiều trắc trở. Đường tâm đạo cho thấy bạn dễ gặp người không phù hợp. Hãy kiên nhẫn và tập trung phát triển bản thân - tình yêu đích thực sẽ đến khi bạn sẵn sàng. Người phù hợp có thể xuất hiện sau tuổi 28.",
            " Đường tình duyên chưa thuận lợi. Bạn có thể trải qua 1-2 mối tình không trọn vẹn trước khi gặp được nửa kia đích thực. Hãy mở lòng hơn và đừng quá cầu toàn."],
        1: [" Tình duyên ổn định và phát triển tốt. Nếu đang có đối tượng, mối quan hệ sẽ bền vững và tiến tới hôn nhân. Người độc thân có cơ hội gặp người phù hợp trong 1-2 năm tới. Bạn là người chung thủy và biết yêu thương.",
            " Vận đào hoa ở mức tốt. Bạn thu hút người khác phái nhờ tính cách ấm áp. Hôn nhân thuận lợi, vợ chồng hòa hợp."],
        2: [" Vận đào hoa rực rỡ! Đường tâm đạo sâu và dài cho thấy bạn là người giàu tình cảm. Tình duyên viên mãn - bạn sẽ gặp được tri kỷ đời mình. Hôn nhân hạnh phúc, gia đình êm ấm, con cái ngoan hiền.",
            " Tình duyên cực kỳ tốt! Bạn có duyên phận tốt, dễ gặp được người yêu thương chân thành. Gia đình hạnh phúc, được bạn đời hết lòng yêu thương."],
    },
    'cong_viec': {
        0: [" Sự nghiệp đang trong giai đoạn thử thách. Có thể gặp khó khăn với cấp trên hoặc thay đổi công việc nhiều lần. Phù hợp với công việc tự do hoặc kinh doanh nhỏ. Thành công sẽ đến sau tuổi 35.",
            " Đường sự nghiệp còn mờ nhạt, bạn chưa tìm được hướng đi rõ ràng. Hãy thử nhiều lĩnh vực để tìm ra đam mê thực sự. Đừng ngại thay đổi."],
        1: [" Công việc ổn định, thu nhập đều đặn. Có cơ hội thăng tiến nếu chủ động học hỏi. Phù hợp với công việc đòi hỏi sự tỉ mỉ như kế toán, kỹ sư, giáo viên. Đạt vị trí quản lý khoảng tuổi 35-40.",
            " Sự nghiệp phát triển ổn định. Được đồng nghiệp tin tưởng và cấp trên đánh giá cao. Có khả năng được cất nhắc trong 2-3 năm tới."],
        2: [" Vận sự nghiệp hanh thông! Đường vận mệnh rõ ràng cho thấy bạn có tố chất lãnh đạo bẩm sinh. Có thể đạt vị trí giám đốc, CEO hoặc chủ doanh nghiệp. Phù hợp với kinh doanh, quản lý, chính trị.",
            " Sự nghiệp rực rỡ! Bạn có khả năng kiếm tiền giỏi và tầm nhìn chiến lược xuất sắc. Sẽ tạo dựng được cơ ngơi vững chắc."],
    },
    'suc_khoe': {
        0: [" Cần chú ý sức khỏe. Đường sinh đạo ngắn KHÔNG có nghĩa tuổi thọ ngắn - mà cho thấy bạn cần chăm sóc bản thân kỹ hơn. Dễ gặp vấn đề về tiêu hóa hoặc stress. Nên tập thể dục 30 phút/ngày, ngủ đủ 7-8 tiếng.",
            " Sức khỏe cần được chú ý đặc biệt. Tránh thức khuya, rượu bia và thức ăn nhiều dầu mỡ. Nên tập yoga hoặc thiền định để cân bằng tâm trí."],
        1: [" Sức khỏe bình thường. Đường sinh đạo cho thấy thể trạng ổn định nhưng cần duy trì lối sống lành mạnh. Chú ý sức khỏe tinh thần. Tuổi thọ trung bình 75-80 tuổi nếu giữ gìn tốt.",
            " Thể trạng tương đối tốt. Bạn có sức đề kháng khá nhưng đôi khi hay mệt mỏi. Duy trì thói quen tập thể dục và ăn uống cân bằng."],
        2: [" Sức khỏe rất tốt! Đường sinh đạo dài, rõ ràng cho thấy thể trạng cường tráng, sức đề kháng mạnh. Tuổi thọ cao (85-95 tuổi), ít bệnh tật. Năng lượng dồi dào, phục hồi nhanh.",
            " Thể trạng xuất sắc! Bạn hiếm khi ốm đau và có sức bền vượt trội. Tuổi già vẫn minh mẫn và khỏe mạnh."],
    },
    'tai_chinh': {
        0: [" Tài chính đang gặp khó khăn. Cần quản lý chi tiêu cẩn thận và tránh đầu tư mạo hiểm. Giai đoạn này phù hợp để tích lũy và tiết kiệm. Vận tài lộc sẽ cải thiện sau tuổi 30.",
            " Đường tài vận chưa rõ ràng. Tiền bạc đến rồi đi, khó tích lũy tài sản lớn ở giai đoạn này. Hãy học cách quản lý tài chính cá nhân."],
        1: [" Tài chính đủ dùng, thu nhập ổn định. Có thể có thêm thu nhập phụ từ đầu tư nhỏ. Đường tài vận cho thấy nên đầu tư vào bất động sản hoặc quỹ tiết kiệm dài hạn.",
            " Tài lộc ở mức tốt. Bạn biết cách kiếm tiền và chi tiêu hợp lý. Có khả năng tích lũy tài sản kha khá vào trung niên."],
        2: [" Tài lộc hanh thông! Đường tài vận sâu cho thấy bạn có khả năng kiếm tiền giỏi. Cơ hội làm giàu từ kinh doanh, đầu tư hoặc bất động sản. Cuộc sống sung túc, nhà lầu xe hơi.",
            " Vận tài lộc cực kỳ tốt! Tiền bạc đến dễ dàng và bền vững. Bạn có tố chất doanh nhân. Gia sản để lại cho con cháu đáng ngưỡng mộ."],
    },
    'tuong_lai': {
        0: [" Tương lai có nhiều biến động nhưng đừng lo - 'mưa dầm thấm lâu'. Những thay đổi sẽ mang đến bài học quý giá. Sau giai đoạn khó khăn (3-5 năm tới) sẽ là những ngày tươi sáng.",
            " Con đường phía trước nhiều ngã rẽ. Bạn có thể phải đối mặt với quyết định quan trọng. Hãy tin vào trực giác. Cuối cùng mọi chuyện sẽ ổn thỏa."],
        1: [" Tương lai ổn định và dần phát triển tốt hơn. Những nỗ lực hiện tại sẽ được đền đáp xứng đáng trong 5-10 năm tới. Giai đoạn 40-50 tuổi là đỉnh cao sự nghiệp.",
            " Triển vọng tương lai tốt. Bạn đang đi đúng hướng và sẽ gặt hái nhiều thành quả. Cuộc sống trung niên và về già đều thoải mái."],
        2: [" Tương lai rực rỡ! Mọi mặt cuộc sống đều phát triển tích cực. Bạn sẽ đạt nhiều thành tựu lớn. Giai đoạn 35-55 tuổi là thời kỳ hoàng kim. Cuối đời an nhàn, con cháu hiếu thảo.",
            " Tương lai vô cùng sáng lạn! Bạn sẽ là người thành đạt và có ảnh hưởng trong cộng đồng. Cuộc sống viên mãn trên mọi phương diện."],
    },
    'kiep_truoc': {
        0: [" Kiếp trước bạn có thể là một người nông dân chăm chỉ, sống cuộc đời giản dị và lương thiện ở vùng quê yên bình. Tuy vất vả nhưng tấm lòng nhân hậu. Nhờ tích đức, kiếp này được hưởng phước lành.",
            " Tiền kiếp bạn là người lao động bình dân, có thể là thợ thủ công hoặc ngư dân. Cuộc sống tuy nghèo nhưng giàu tình nghĩa. Công đức giúp người mang lại phước kiếp này."],
        1: [" Kiếp trước bạn có thể là một thương nhân giàu có hoặc học giả uyên bác. Được nhiều người tôn trọng. Trí tuệ từ kiếp trước giúp bạn nhanh nhạy trong kiếp này.",
            " Tiền kiếp bạn có thể là thầy thuốc hoặc thầy giáo, chuyên giúp đời cứu người. Kiến thức tích lũy mang lại sự thông minh trong kiếp này."],
        2: [" Kiếp trước bạn có thể là người có địa vị cao - quý tộc, quan lại hoặc người tu hành đắc đạo. Công đức tích lũy từ nhiều kiếp mang lại vận mệnh tốt đẹp. Phong thái quý phái, uy nghi tự nhiên.",
            " Tiền kiếp bạn là người thuộc tầng lớp thượng lưu, có thể là hoàng gia hoặc cao tăng. Phước đức sâu dày, kiếp này được hưởng cuộc sống đầy đủ."],
    },
    'kiep_sau': {
        0: [" Kiếp sau bạn sẽ được sinh ra trong gia đình ấm cúng ở vùng quê thanh bình. Cuộc sống giản dị nhưng tràn đầy hạnh phúc. Bạn sẽ sống gần gũi với thiên nhiên, tâm hồn thanh thản.",
            " Lai sinh bạn sẽ có cuộc sống bình dị nhưng an yên. Được bao quanh bởi người yêu thương. Tuy không giàu nhưng luôn đủ đầy và hạnh phúc."],
        1: [" Kiếp sau bạn sẽ có cuộc sống khá giả, được sinh ra trong gia đình có điều kiện tốt. Có cơ hội học hành cao, đạt thành công trong sự nghiệp. Cuộc sống thoải mái, nhiều niềm vui.",
            " Lai sinh bạn sẽ được tái sinh vào gia đình trung lưu thịnh vượng. Có điều kiện phát triển tài năng. Sự nghiệp thành đạt, gia đình hạnh phúc."],
        2: [" Kiếp sau bạn sẽ được hưởng phú quý vinh hoa! Nhờ công đức tích lũy, bạn sẽ được sinh ra trong gia đình danh giá. Cuộc sống sung túc, tài năng xuất chúng, được ngưỡng mộ.",
            " Lai sinh bạn sẽ được đầu thai vào cảnh phú quý, gia đình giàu sang. Thông minh từ nhỏ, tài năng vượt trội. Cuộc đời nhiều vinh quang và thành tựu."],
    },
}

LEVEL_NAMES = {
    'fortune_class': ['Bình thường ', 'Tốt ', 'Rất tốt '],
    'tinh_duyen':    ['Chưa thuận lợi ', 'Bình ổn ', 'Rất tốt '],
    'cong_viec':     ['Nhiều thử thách ', 'Ổn định ', 'Hanh thông '],
    'suc_khoe':      ['Cần chú ý ', 'Bình thường ', 'Rất tốt '],
    'tai_chinh':     ['Khó khăn ', 'Đủ dùng ', 'Sung túc '],
    'tuong_lai':     ['Nhiều biến động ', 'Ổn định ', 'Rực rỡ '],
    'kiep_truoc':    ['Người bình dân ', 'Thương nhân/Học giả ', 'Quý tộc/Cao tăng '],
    'kiep_sau':      ['Cuộc sống giản dị ', 'Cuộc sống khá giả ', 'Phú quý vinh hoa '],
}

print(" Đã load từ điển dự đoán tử vi chi tiết!")

In [ ]:
def xem_tu_vi(model, image_path, image_size=(128, 128)):
    """
     XEM TỬ VI BẰNG CHỈ TAY
    Phân tích ảnh chỉ tay và dự đoán chi tiết 8 khía cạnh cuộc sống.
    """
    import random
    
    img = cv2.imread(image_path)
    if img is None:
        print(f" Không thể đọc ảnh: {image_path}")
        return
    
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, image_size)
    img_normalized = img_resized.astype(np.float32) / 255.0
    img_batch = np.expand_dims(img_normalized, axis=0)
    
    predictions = model.predict(img_batch, verbose=0)
    
    # === HIỂN THỊ ẢNH + BIỂU ĐỒ ===
    fig = plt.figure(figsize=(20, 6))
    
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(img_rgb)
    ax1.set_title(' Ảnh chỉ tay', fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Biểu đồ radar
    ax2 = fig.add_subplot(1, 3, 2, polar=True)
    aspects = ['Tổng quan', 'Tình duyên', 'Công việc', 'Sức khỏe',
               'Tài chính', 'Tương lai', 'Kiếp trước', 'Kiếp sau']
    
    pred_levels = []
    for name in OUTPUT_NAMES:
        proba = predictions[name][0]
        score = proba[0]*0 + proba[1]*1 + proba[2]*2
        pred_levels.append(score)
    
    angles = np.linspace(0, 2*np.pi, len(aspects), endpoint=False).tolist()
    pred_plot = pred_levels + [pred_levels[0]]
    angles_plot = angles + [angles[0]]
    
    ax2.fill(angles_plot, pred_plot, alpha=0.25, color='#FF6B6B')
    ax2.plot(angles_plot, pred_plot, 'o-', linewidth=2, color='#FF6B6B')
    ax2.set_xticks(angles)
    ax2.set_xticklabels(aspects, fontsize=8)
    ax2.set_ylim(0, 2)
    ax2.set_title('Biểu đồ Vận Mệnh', fontsize=14, fontweight='bold', pad=20)
    
    # Biểu đồ xác suất fortune_class
    ax3 = fig.add_subplot(1, 3, 3)
    fc_proba = predictions['fortune_class'][0]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    bars = ax3.barh(['Bình thường', 'Tốt', 'Rất tốt'], fc_proba * 100,
                    color=colors, edgecolor='black', linewidth=0.5)
    ax3.set_xlim(0, 100)
    ax3.set_xlabel('Xác suất (%)')
    ax3.set_title('Xác suất Tổng Quan', fontsize=14, fontweight='bold')
    for bar, prob in zip(bars, fc_proba):
        ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{prob*100:.1f}%', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    # === IN KẾT QUẢ CHI TIẾT ===
    print("\n" + "═" * 70)
    print("         KẾT QUẢ XEM TỬ VI BẰNG CHỈ TAY ")
    print("═" * 70)
    print(f"  📁 File: {os.path.basename(image_path)}")
    print("═" * 70)
    
    for name in OUTPUT_NAMES:
        proba = predictions[name][0]
        pred_class = int(np.argmax(proba))
        confidence = proba[pred_class] * 100
        label_vn = OUTPUT_LABELS_VN[name]
        level_name = LEVEL_NAMES[name][pred_class]
        description = random.choice(PREDICTIONS_TEXT[name][pred_class])
        
        print(f"\n{'─' * 70}")
        print(f"  {label_vn}")
        print(f"  Kết quả: {level_name} (Độ tin cậy: {confidence:.1f}%)")
        print(f"  {'─' * 60}")
        # Wrap text
        words = description.split()
        line = "  "
        for word in words:
            if len(line) + len(word) > 68:
                print(line)
                line = "  " + word + " "
            else:
                line += word + " "
        if line.strip():
            print(line)
    
    # Tổng kết
    fc_pred = int(np.argmax(predictions['fortune_class'][0]))
    overall = [' BÌNH THƯỜNG', ' TỐT', ' RẤT TỐT'][fc_pred]
    
    print(f"\n{'═' * 70}")
    print(f"   TỔNG KẾT VẬN MỆNH: {overall}")
    
    total_score = 0
    for name in OUTPUT_NAMES:
        proba = predictions[name][0]
        score = (proba[0]*0 + proba[1]*50 + proba[2]*100)
        total_score += score
    avg_score = total_score / len(OUTPUT_NAMES)
    
    print(f"   Điểm vận mệnh tổng hợp: {avg_score:.0f}/100")
    if avg_score >= 70:
        print("   Chúc mừng! Bạn có vận mệnh rất tốt!")
    elif avg_score >= 40:
        print("   Vận mệnh tốt, hãy tiếp tục phát huy!")
    else:
        print("   Hãy kiên nhẫn, mọi chuyện sẽ tốt đẹp hơn!")
    print("═" * 70)
    print("   Lưu ý: Kết quả chỉ mang tính chất giải trí và tham khảo.")
    print("═" * 70)

## 10.  Thử Xem Tử Vi!

In [ ]:
# Lấy 3 ảnh đầu tiên để thử
test_files = sorted([f for f in os.listdir(DATASET_PATH) if f.lower().endswith(('.jpg','.jpeg','.png'))])[:3]

for fname in test_files:
    img_path = os.path.join(DATASET_PATH, fname)
    xem_tu_vi(model, img_path)
    print("\n\n")

## 11. Lưu mô hình

In [ ]:
model_save_path = os.path.join(DATASET_PATH, 'palmistry_multioutput_model.keras')
if 'model' in locals():
    model.save(model_save_path)
    print(f" Đã lưu mô hình tại: {model_save_path}")
    
    print(f"\n Thông tin mô hình:")
    print(f"  - Kiến trúc: CNN Multi-Output (Functional API)")
    print(f"  - Tổng tham số: {model.count_params():,}")
    print(f"  - Kích thước ảnh: {IMAGE_SIZE}")
    print(f"  - Số output heads: {len(OUTPUT_NAMES)}")
    print(f"  - Outputs: {OUTPUT_NAMES}")

## 12. Hướng dẫn sử dụng

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════╗
║      HƯỚNG DẪN SỬ DỤNG MÔ HÌNH XEM TỬ VI CHỈ TAY     ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1. Load mô hình đã lưu:                                    ║
║     from tensorflow.keras.models import load_model           ║
║     model = load_model('palmistry_multioutput_model.keras')  ║
║                                                              ║
║  2. Xem tử vi từ ảnh chỉ tay:                               ║
║     xem_tu_vi(model, 'đường/dẫn/tới/ảnh.jpg')               ║
║                                                              ║
║  3. Mô hình dự đoán 8 khía cạnh:                            ║
║      Tổng quan vận mệnh                                   ║
║      Tình duyên & hôn nhân                                 ║
║      Công việc & sự nghiệp                                 ║
║      Sức khỏe & tuổi thọ                                   ║
║      Tài chính & tài lộc                                   ║
║      Tương lai                                             ║
║       Kiếp trước                                          ║
║      Kiếp sau                                              ║
║                                                              ║
║   Lưu ý: Kết quả chỉ mang tính giải trí và tham khảo.   ║
╚══════════════════════════════════════════════════════════════╝
""")